# Tutorial 04 — Selecting fluid systems

Real documents mix systems: a single file may hold Methanol+Water *and* Ethanol+Water rows, or a
ternary where you only want one sub-ratio. `fairfluids.analysis.fluids` (alias **`flu`**) turns
"which system" into a composable **row mask** over the `mole_fraction_<compound>` columns.

A selector is just a callable `selector(df) -> boolean mask`, so it composes with `& | ~` and drops
straight into `BayesianDataset.from_documents(..., row_filter=...)`.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import numpy as np
import pandas as pd
import sympy as sp

# Absolute data paths so the notebook runs from anywhere.
ROOT = "/home/sga/Code/FAIRFluids"
DENSITY = os.path.join(ROOT, "transition_water", "data", "Density")
VISCOSITY = os.path.join(ROOT, "transition_water", "data", "Viscosity")

from fairfluids import FAIRFluidsDocument
from fairfluids.analysis import fluids as flu
from fairfluids.analysis import extract_property_dataframe

corpus = {
    "Methanol/Ethanol + Water": os.path.join(DENSITY, "Methanol_water_10.1021_je700300y.json"),
    "Glycerol + Water":         os.path.join(DENSITY, "WaterGlycerol_egorov_density.json"),
    "Urea + Water":             os.path.join(DENSITY, "UreaWater_jct_density.json"),
}
docs = {k: FAIRFluidsDocument.model_validate_json(open(v).read()) for k, v in corpus.items()}

frames = [extract_property_dataframe(d, property_type="density",
                                     keep_only_relevant_columns=False) for d in docs.values()]
big = pd.concat(frames, ignore_index=True)
print("total rows:", len(big))

total rows: 968


## 1. The building blocks

* `flu.system(*compounds)` — rows whose fluid system is **exactly** that set of compounds.
* `flu.contains(compound)` — rows that include a given compound (system may be larger).
* `flu.where(**ranges)` — inclusive range filter on `mole_fraction_<compound>` columns.
* `flu.system(...).ratio(**parts)` — pin a fixed ratio *within* named components (others free).

In [2]:
def systems_kept(selector):
    sub = big[selector(big)]
    return sorted({tuple(sorted(c)) for c in sub["fluid_compounds"].dropna()
                   if isinstance(c, list)})

print("everything             :", systems_kept(lambda df: pd.Series(True, index=df.index)))
print("exact Methanol + Water :", systems_kept(flu.system("Methanol", "Water")))
print("exact Ethanol + Water  :", systems_kept(flu.system("Ethanol", "Water")))
print("contains Urea          :", systems_kept(flu.contains("Urea")))

everything             : [('Ethanol',), ('Ethanol', 'Methanol'), ('Ethanol', 'Methanol', 'Water'), ('Ethanol', 'Water'), ('Glycerol', 'Water'), ('Methanol',), ('Methanol', 'Water'), ('Urea', 'Water')]
exact Methanol + Water : [('Methanol', 'Water')]
exact Ethanol + Water  : [('Ethanol', 'Water')]
contains Urea          : [('Urea', 'Water')]


## 2. Compose with `& | ~`

Selectors are boolean-algebra citizens: union (`|`), intersection (`&`), negation (`~`).

In [3]:
union = flu.system("Methanol", "Water") | flu.system("Glycerol", "Water")
print("Methanol | Glycerol    :", systems_kept(union))

not_urea = ~flu.contains("Urea")
print("everything but Urea    :", systems_kept(not_urea))

Methanol | Glycerol    : [('Glycerol', 'Water'), ('Methanol', 'Water')]
everything but Urea    : [('Ethanol',), ('Ethanol', 'Methanol'), ('Ethanol', 'Methanol', 'Water'), ('Ethanol', 'Water'), ('Glycerol', 'Water'), ('Methanol',), ('Methanol', 'Water')]


## 3. Pin a sub-ratio while a third component varies

For a ternary you might want Glycerol:Methanol fixed at **1:2** while water is free. `.ratio(...)`
normalises *within* the named components, so water can take any value. (Shown on a tiny synthetic
frame, since the density files above are binaries.)

In [4]:
demo = pd.DataFrame({
    "mole_fraction_glycerol": [0.10, 0.20, 0.05, 0.00],
    "mole_fraction_methanol": [0.20, 0.30, 0.10, 0.40],
    "mole_fraction_water":    [0.70, 0.50, 0.85, 0.60],
    "label": ["G:M = 1:2", "G:M = 2:3", "G:M = 1:2", "no glycerol"],
})
sel = flu.system("glycerol", "methanol", "water").ratio(glycerol=1, methanol=2)
print("kept (G:M = 1:2, water free):", demo[sel(demo)]["label"].tolist())

kept (G:M = 1:2, water free): ['G:M = 1:2', 'G:M = 1:2']


## 4. Plug a selector into dataset construction

`row_filter` accepts any selector and is applied after extraction, before grouping — a strict
superset of the older `composition_filter`.

In [5]:
from fairfluids.analysis.bayesian import BayesianDataset

methanol_water = BayesianDataset.from_documents(
    docs, property="density", features=["temperature"],
    group_by=["source_doi", "mole_fraction_water"], min_points=4,
    row_filter=flu.system("Methanol", "Water"),
)
print("Methanol+Water groups kept:", len(methanol_water))
methanol_water.to_overview().head()

Methanol+Water groups kept: 12


/home/sga/Code/FAIRFluids/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,group_label,n_points,has_uncertainty,source_doi,mole_fraction_water,temperature_min,temperature_max,obs_min,obs_max
0,10.1021/je700300y | 0.0871,7,True,10.1021/je700300y,0.0871,283.15,313.15,6.668889,6.703139
1,10.1021/je700300y | 0.1529,7,True,10.1021/je700300y,0.1529,283.15,313.15,6.683874,6.716909
2,10.1021/je700300y | 0.2395,7,True,10.1021/je700300y,0.2395,283.15,313.15,6.703973,6.735542
3,10.1021/je700300y | 0.3306,7,True,10.1021/je700300y,0.3306,283.15,313.15,6.725766,6.755781
4,10.1021/je700300y | 0.4064,7,True,10.1021/je700300y,0.4064,283.15,313.15,6.744212,6.772886


## Recap

* `flu.system / contains / where / ratio` build readable row masks over composition columns.
* They compose with `& | ~` and are plain callables, so they slot into
  `BayesianDataset.from_documents(row_filter=...)` and any pandas filtering.
* `.ratio(...)` pins a ratio *within* named components while leaving the rest free — ideal for
  isolating one cut of a ternary.

That completes the analysis-package tour: **declare** (T01) → **fit frequentist** (T02) →
**fit Bayesian** (T03) → **select systems** (T04).